In [11]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow 
from tensorflow import keras
# from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, BatchNormalization
# from keras.models import Model
# from keras.optimizers import Adam
# from keras import backend as K

layers = keras.layers
Adam = keras.optimizers.Adam
k = keras.backend

# --- KONFIGURASI FOLDER ---
TRAIN_PATH = 'dataset/train/images'
MASK_PATH = 'dataset/train/mask'
TEST_PATH = 'dataset/test/images'

# --- SETTING ---
IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3

# --- FUNGSI DICE LOSS (DIPERBAIKI) ---
def dice_coef(y_true, y_pred):
    # FIX: Pastikan y_true berupa angka (float32), bukan True/False
    y_true = keras.cast(y_true, keras.float32)
    y_pred = keras.cast(y_pred, keras.float32)
    
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + 1) / (K.sum(y_true_f) + K.sum(y_pred_f) + 1)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

# --- FUNGSI RLE ---
def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

# --- 1. LOAD DATA TRAINING ---
print("1. Sedang membaca data training (Mode 256x256)...")

all_files = sorted(os.listdir(TRAIN_PATH))
train_ids = [f for f in all_files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

X_train = []
Y_train = []

print(f"   Mendeteksi {len(train_ids)} gambar. Mulai loading...")

for filename in train_ids:
    path_img = os.path.join(TRAIN_PATH, filename)
    img = cv2.imread(path_img)
    if img is None: continue
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    
    # Ganti nama train -> mask
    mask_name = filename.replace('train', 'mask').replace('.jpg', '.png').replace('.jpeg', '.png')
    path_mask = os.path.join(MASK_PATH, mask_name)
    
    mask = cv2.imread(path_mask, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"   [SKIP] Gagal menemukan mask: {mask_name}")
        continue
    
    mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT))
    
    X_train.append(img)
    
    # FIX DI SINI: Ubah Boolean (True/False) jadi Float (1.0/0.0)
    mask_bool = (np.expand_dims(mask, axis=-1) > 128).astype(np.float32)
    Y_train.append(mask_bool)

X_train = np.array(X_train)
Y_train = np.array(Y_train)

print(f"   SUKSES! Berhasil load {len(X_train)} pasang data.")

# --- 2. BUAT MODEL U-NET ---
print("2. Membangun Model U-Net Versi Upgrade...")

def conv_block(inputs, filters):
    x = Conv2D(filters, (3, 3), padding="same", activation="relu")(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    return x

inputs = Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
s = keras.keras.layers.Lambda(lambda x: x / 255)(inputs)

# Encoder
c1 = conv_block(s, 16)
p1 = MaxPooling2D((2, 2))(c1)
p1 = Dropout(0.1)(p1)

c2 = conv_block(p1, 32)
p2 = MaxPooling2D((2, 2))(c2)
p2 = Dropout(0.1)(p2)

c3 = conv_block(p2, 64)
p3 = MaxPooling2D((2, 2))(c3)
p3 = Dropout(0.2)(p3)

c4 = conv_block(p3, 128)
p4 = MaxPooling2D(pool_size=(2, 2))(c4)
p4 = Dropout(0.2)(p4)

# Bridge
c5 = conv_block(p4, 256)

# Decoder
u6 = UpSampling2D((2, 2))(c5)
u6 = concatenate([u6, c4])
c6 = conv_block(u6, 128)

u7 = UpSampling2D((2, 2))(c6)
u7 = concatenate([u7, c3])
c7 = conv_block(u7, 64)

u8 = UpSampling2D((2, 2))(c7)
u8 = concatenate([u8, c2])
c8 = conv_block(u8, 32)

u9 = UpSampling2D((2, 2))(c8)
u9 = concatenate([u9, c1])
c9 = conv_block(u9, 16)

outputs = Conv2D(1, (1, 1), activation='sigmoid')(c9)

model = Model(inputs=[inputs], outputs=[outputs])

# Compile dengan dice_loss
model.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=[dice_coef, 'accuracy'])

# --- 3. TRAINING ---
print("3. Mulai Training (Epochs 50 - Tunggu ya)...")
model.fit(X_train, Y_train, validation_split=0.1, batch_size=8, epochs=50)
print("   Training selesai!")

# --- 4. PREDIKSI DATA TEST ---
print("4. Membuat file submission...")
test_files = sorted(os.listdir(TEST_PATH))
test_files = [f for f in test_files if f.lower().endswith(('.jpg', '.png'))]
submission_data = []

for filename in test_files:
    path = os.path.join(TEST_PATH, filename)
    img_orig = cv2.imread(path)
    if img_orig is None: continue
    
    h_orig, w_orig = img_orig.shape[:2]
    
    # Resize & Prediksi
    img_input = cv2.resize(img_orig, (IMG_WIDTH, IMG_HEIGHT))
    img_input = np.expand_dims(img_input, axis=0)
    
    pred = model.predict(img_input, verbose=0)[0]
    
    # Resize Balik
    pred_resized = cv2.resize(pred, (w_orig, h_orig))
    
    # Threshold 0.5 (Kembali ke standar, bisa diturunkan jika perlu)
    mask_final = np.where(pred_resized > 0.5, 255, 0).astype(np.uint8)
    
    if np.max(mask_final) == 255:
        rle = encode_rle(mask_final)
    else:
        rle = ""
    submission_data.append([filename, rle])

df = pd.DataFrame(submission_data, columns=['ImageId', 'rle'])
df.to_csv('submission_v2_fix.csv', index=False)
print("\nSELESAI! Cek file 'submission_v2_fix.csv'.")

VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow/core/framework/attr_value.proto: gencode 5.28.3 runtime 5.28.0. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.